# Agentic demo — evolving a system prompt for a real tool-using agent

**The agentic counterpart to the GEPA/SkillOpt demos.** Here the rollout is a *real
multi-turn `claude -p` agent* with a **Bash tool** — it thinks, runs a shell command,
reads the result, and answers (`num_turns` > 1). So the `Trace`s carry real
`tool_use`/`tool_result` pairs, not single-shot Q&A.

This is the first **driver-mode** optimizer: it runs through `agentbook.loop.run_iteration`
arrow by arrow (rollout → evaluate → reflect → edit → gate), unlike the engine-mode
GEPA/SkillOpt adapters that own their own loop.

> **Cost note:** every rollout shells out to `claude -p` (a real Bedrock call, ~$0.2 each
> with the local CLI's system prompt). The eval set is kept tiny on purpose.

In [ ]:
import os

from utils import bootstrap

REPO = bootstrap()  # repo root, .env, src on path (no hardcoded path)

from agentbook.adapters.claude_agent_adapter import ClaudeAgentOptimizer
from agentbook.loop import run_iteration
from agentbook.session import Session

KERNEL_PID = os.getpid()
print("kernel PID:", KERNEL_PID, "| claude on PATH:", bool(os.environ.get("PATH")))

## Step 1 — a tiny eval set that *forces* tool use

Each task can only be answered correctly by running a shell command, and each has a
deterministic gold answer — so `evaluate` is exact (substring) matching, and a passing
trace necessarily contains a real `Bash` tool call.

In [ ]:
EVAL = [
    {"id": "mul", "prompt": "Use the Bash tool to compute 17 * 23, then reply with only the number.", "gold": "391"},
    {
        "id": "sha",
        "prompt": "Use the Bash tool to compute the SHA256 of the exact string foo "
        "(no trailing newline). Reply with only the 64-hex-char digest.",
        "gold": "2c26b46b68ffc68ff99b453c1d30413413422d706483bfa0f98a5e886266e7ae",
    },
]

seed_prompt = "You are a helpful assistant."
session = Session(
    eval_set=EVAL, model_client=lambda *a, **k: None, slice_kind="system_prompt", seed_artifact=seed_prompt
)
opt = ClaudeAgentOptimizer(session, model="claude-haiku-4-5", reflect_model="claude-sonnet-4-6")
print("eval tasks:", [e["id"] for e in EVAL], "| slice:", session.slice_kind)

## Step 2 — run one driver-mode iteration (live)

`run_iteration` rolls out the seed prompt over the eval set, scores it, reflects an
improved system prompt, rolls *that* out, and keeps it only if it scores at least as
well (`gate`). Each rollout is a real agent episode.

In [ ]:
it = run_iteration(opt, session)
print("kernel PID stable:", os.getpid() == KERNEL_PID)
print("best score      :", session.best_score())
for tid, res in opt.last_results.items():
    print(
        f"  task {tid}: answer={res.answer[:20]!r} tools={[t.name for t in res.tool_calls]} "
        f"turns={res.num_turns} cost=${res.cost_usd:.3f}"
    )

## Analysis — genuinely agentic this time

- **Real tool use:** each rollout's `Trace.signals["tool_calls"]` lists the `Bash` calls the
  agent made; `num_turns` > 1 confirms it's multi-turn (think → tool → answer), not single-shot.
- **The slice under evolution is still the system prompt**, but the *task* is now an agent
  episode with a tool, scored by task success — closer to "optimizing an agent" than the
  AIME/SearchQA Q&A demos.
- **Tokens & cost are real here:** parsed from the `result` event's `usage`/`total_cost_usd`
  (the `claude -p` path exposes them, unlike the litellm GEPA path).

## Data sources

Every value comes from a real source — no fabricated rows (Constitution I):

- **Agent:** local `claude -p --output-format stream-json --allowedTools Bash` (real Bedrock calls).
- **Eval set:** the inline tool-forcing tasks above, with deterministic gold answers verifiable by hand (`17*23=391`; `sha256("foo")=2c26b4…e7ae`).
- **Scores/cost/tokens:** parsed live from each run's stream-json `result` event.